# Omega Safe SeqEdit Real-World Benchmark

This notebook is intentionally thin. It calls scripts and loads saved JSON outputs. Core logic lives in `src/omega_safe_seqedit`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
print(REPO)
sys.path.insert(0, str(REPO / 'src'))

## Choose Preset

Use `mac_debug` first. For a real run, edit `configs/local_real_mac.yaml` or `configs/hg002_chr20_mac.yaml` and switch `PRESET`.

In [ ]:
PRESET = 'mac_debug'  # 'local_real_mac' or 'hg002_chr20_mac'
RUN_PREPROCESS = True
RUN_BASELINES = True
RUN_TARGET_ONLY = True
RUN_FULL = True
RUN_EVALUATION = True
CONFIG = REPO / 'configs' / f'{PRESET}.yaml'
CONFIG

In [ ]:
def run_cmd(args):
    print(' '.join(str(a) for a in args))
    return subprocess.run(args, cwd=REPO, check=True)

if RUN_PREPROCESS:
    run_cmd([sys.executable, 'scripts/preprocess_dataset.py', '--config', str(CONFIG)])

## Baselines

Runs no-edit, conservative consensus, support-rule, and any external FASTA/FASTQ baselines listed in the config.

In [ ]:
if RUN_BASELINES:
    run_cmd([sys.executable, 'scripts/run_baselines.py', '--config', str(CONFIG), '--split', 'test'])

## Train SeqEdit Models

`target_only` measures sequence-context priors. `full` adds support/pileup/rule features.

In [ ]:
if RUN_TARGET_ONLY:
    run_cmd([sys.executable, 'scripts/train_model.py', '--config', str(CONFIG), '--run', 'target_only'])
if RUN_FULL:
    run_cmd([sys.executable, 'scripts/train_model.py', '--config', str(CONFIG), '--run', 'full'])

## Evaluate Neural, Rule, and Hybrid Decoding

Hybrid is the safety-first production/debug mode. Neural-only reveals whether the model itself learned the edit decision without rule forcing.

In [ ]:
if RUN_EVALUATION:
    eval_grid = [('target_only', 'neural'), ('full', 'neural'), ('full', 'hybrid')]
    for run_name, mode in eval_grid:
        ckpt = REPO / 'outputs' / PRESET / 'runs' / run_name / 'best.ckpt'
        if ckpt.exists():
            run_cmd([sys.executable, 'scripts/evaluate_predictions.py', '--config', str(CONFIG), '--run', run_name, '--split', 'test', '--mode', mode])

## Compare Summaries

In [ ]:
from omega_safe_seqedit.config import load_config
cfg = load_config(CONFIG)
out = Path(cfg['paths']['output_dir'])
rows = []
baseline_path = out / 'baselines' / 'test_baseline_summary.json'
if baseline_path.exists():
    for name, summary in json.loads(baseline_path.read_text()).items():
        rows.append({'name': name, **{k: summary.get(k) for k in ['usable_score','identity','overcorrection_rate','hard_edit_false_positive_rate','corrected_edits','missed_edits','false_edits']}})
for path in sorted((out / 'runs').glob('*/*_summary.json')) if (out / 'runs').exists() else []:
    summary = json.loads(path.read_text())
    rows.append({'name': '/'.join(path.relative_to(out).parts[:-1]) + '/' + path.stem, **{k: summary.get(k) for k in ['usable_score','identity','overcorrection_rate','hard_edit_false_positive_rate','corrected_edits','missed_edits','false_edits']}})
rows

## False-Edit Audit

The table below is the first thing to inspect before tuning thresholds. It is the safety lens.

In [ ]:
audit_rows = []
for path in sorted((out / 'runs').glob('*/*_summary.json')) if (out / 'runs').exists() else []:
    summary = json.loads(path.read_text())
    for item in summary.get('false_edit_table', [])[:50]:
        audit_rows.append({'source': str(path.relative_to(out)), **item})
audit_rows[:50]

## Qualitative Examples

In [ ]:
pred_path = out / 'runs' / 'full' / 'test_hybrid_predictions.jsonl'
if pred_path.exists():
    examples = [json.loads(line) for line in pred_path.read_text().splitlines() if line.strip()]
    for ex in examples[:3]:
        print('\n---', ex['example_id'], ex.get('case_type'))
        print('target    ', ex['target_seq'][:160])
        print('truth     ', ex['truth_seq'][:160])
        print('prediction', ex['prediction'][:160])
        print('pred_events', ex.get('pred_events', [])[:12])
        print('trace_sample', ex.get('trace', [])[:3])